<a href="https://colab.research.google.com/github/danielcarvalhoa113-stack/AED2---FATEC-Votorantim/blob/main/Aula06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lista 06, pilhas.<br> focado nos projetos práticos

Projeto 1

In [3]:
import pandas as pd

# Implementação simples de uma Pilha (Stack) usando a lista do Python
class Stack:
    def __init__(self):
        self.itens = []

    def push(self, item):
        self.itens.append(item)

    def pop(self):
        if not self.is_empty():
            return self.itens.pop()
        return None

    def is_empty(self):
        return len(self.itens) == 0

class HistoricoOperacoes:
    def __init__(self, dataframe):
        self.estado_atual = dataframe.copy()
        self.pilha_estados = Stack() # Agora a classe Stack existe!

    def aplicar_filtro(self, condicao):
        self.pilha_estados.push(self.estado_atual.copy())
        self.estado_atual = self.estado_atual.loc[condicao].copy()

    def desfazer(self):
        if self.pilha_estados.is_empty():
            print("Nenhum estado anterior para desfazer.")
            return False

        self.estado_atual = self.pilha_estados.pop()
        return True

    def mostrar(self):
        print(self.estado_atual.to_string(index=False))

# Exemplo de uso
df = pd.DataFrame({
    "produto": ["Notebook", "Mouse", "Monitor", "Teclado"],
    "vendas": [3200, 800, 2100, 1200]
})

historico = HistoricoOperacoes(df)

historico.aplicar_filtro(historico.estado_atual["vendas"] >= 1000)
print("Após o filtro:")
historico.mostrar()

historico.desfazer()
print("\nApós desfazer:")
historico.mostrar()

Após o filtro:
 produto  vendas
Notebook    3200
 Monitor    2100
 Teclado    1200

Após desfazer:
 produto  vendas
Notebook    3200
   Mouse     800
 Monitor    2100
 Teclado    1200


Projeto 2

In [5]:
import re

# Implementação da Pilha (Stack) com o método 'peek' adicionado
class Stack:
    def __init__(self):
        self.itens = []

    def push(self, item):
        self.itens.append(item)

    def pop(self):
        if not self.is_empty():
            return self.itens.pop()
        return None

    def peek(self):
        if not self.is_empty():
            return self.itens[-1]
        return None

    def is_empty(self):
        return len(self.itens) == 0

def avaliar_expressao(expressao):
    precedencia = {'+': 1, '-': 1, '*': 2, '/': 2}
    valores = Stack()
    operadores = Stack()

    tokens = re.findall(r'\d+(?:\.\d+)?|[()+\-*/]', expressao.replace(' ', ''))

    def aplicar_operador():
        operador = operadores.pop()
        b = valores.pop()
        a = valores.pop()

        if a is None or b is None:
            raise ValueError("Expressão inválida")

        if operador == '+':
            valores.push(a + b)
        elif operador == '-':
            valores.push(a - b)
        elif operador == '*':
            valores.push(a * b)
        elif operador == '/':
            if b == 0:
                raise ZeroDivisionError("Divisão por zero")
            valores.push(a / b)

    for token in tokens:
        if token.replace('.', '', 1).isdigit():
            valores.push(float(token) if '.' in token else int(token))

        elif token == '(':
            operadores.push(token)

        elif token == ')':
            while not operadores.is_empty() and operadores.peek() != '(':
                aplicar_operador()

            if operadores.is_empty():
                raise ValueError("Parênteses desbalanceados")
            operadores.pop()

        else:
            while (not operadores.is_empty()
                   and operadores.peek() != '('
                   and precedencia[operadores.peek()] >= precedencia[token]):
                aplicar_operador()
            operadores.push(token)

    while not operadores.is_empty():
        if operadores.peek() == '(':
            raise ValueError("Parênteses desbalanceados")
        aplicar_operador()

    resultado = valores.pop()
    if not valores.is_empty():
        raise ValueError("Expressão inválida")
    return resultado

print(avaliar_expressao("3 + 4 * (2 - 1)"))  # Deve retornar 7.0 (ou 7)
print(avaliar_expressao("10 / 2 + 6 * 3"))   # Deve retornar 23.0 (ou 23)

7
23.0
